#CSV Ingestion via Delta Live Tables (Chunk 2)

In [0]:
# DLT Pipeline Script
import dlt
from pyspark.sql.functions import current_timestamp, lit

@dlt.table(
    name="listings_csv_dlt",
    comment="Bronze: Ingesting Chunk 2 CSV with strict path filtering",
    table_properties={
        "quality": "bronze",
        "delta.enableChangeDataFeed": "true"
    }
)
def listings_csv_dlt():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        # CRITICAL: pathGlobFilter ensures we ONLY look at CSV files
        # This prevents Spark from trying to read .xml or .json as CSV
        .option("pathGlobFilter", "*.csv") 
        .option("cloudFiles.inferColumnTypes", "false") 
        .load("/Volumes/vstone_catalog/raw/chunks/")
        # Further filter for specifically Chunk 2
        .filter("_metadata.file_name = '1_main_chunk_2.csv'") 
        .withColumn("load_dt", current_timestamp())
        .withColumn("source_file", lit("1_main_chunk_2.csv"))
    )